# Lab 2 – Reuse and Orchestrate AI Agents with Microsoft Foundry SDK (Python)

## Scenario
You will reuse the **ProductInventoryManager** agent created in Lab 1 and interact with it from **Visual Studio Code** using **Python**. Then you will create a second agent and orchestrate both agents using the **Microsoft Agent Framework** workflow pattern.

> **Important:** In the new Foundry Agents architecture, agent behavior is defined at creation time (versions are static). Avoid trying to override tools/instructions at runtime.

## Prerequisites
- Completed **Lab 1** (you have **Project Endpoint** + **ProductInventoryManager Agent ID**)
- Visual Studio Code + Microsoft Foundry extension installed
- Python 3.10+
- Azure CLI installed and authenticated: `az login`

### Values you need from Lab 1
- `AZURE_FOUNDRY_PROJECT_ENDPOINT`
- `PRODUCT_INVENTORY_MANAGER_AGENT_ID`

## Step 1 — (Optional) Connect VS Code to Foundry
1. Install the **Microsoft Foundry for VS Code** extension.
2. Sign in to Azure.
3. Set your **default Foundry project** so Agents/Models appear in the sidebar.

This step validates you are pointing at the same Foundry project as Lab 1.

## Step 3 — Install required packages
These packages are used in Microsoft Foundry quickstarts and SDK guidance for working with Foundry projects and agents.# Create your chat client and agents
deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", model_id)
chat_client = AzureOpenAIChatClient(
    credential=AzureCliCredential(),
    deployment_name=deployment_name,
)


In [1]:
%pip install azure-ai-projects --pre
%pip install azure-identity openai python-dotenv

# Microsoft Agent Framework (Python)
%pip install agent-framework --pre

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 2.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.2 MB/s eta 0:00:00a 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.5/434.5 kB 8.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.1/120.1 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 16.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 12.6 MB/s eta 0:00:00
     

## Step 4 — Configure environment variables
Create a `.env` file in this folder with the following values.

In [2]:
# Create/overwrite .env with your values
# (You can also set these in your shell instead.)

#AZURE_FOUNDRY_PROJECT_ENDPOINT="<paste-your-project-endpoint-here>"
#PRODUCT_INVENTORY_MANAGER_AGENT_ID="<paste-agent-id-here>"

# Optional if you create a new agent in this lab:
#AZURE_FOUNDRY_PROJECT_MODEL_ID="gpt-4o-mini"

## Step 5 — Authenticate to Azure
Make sure you are logged in before running Python.

In [3]:
# Run in terminal (not in notebook)
# if you haven't already authenticated with Azure CLI, do so now with:
# if you get error like comand az not found then run below in bash
# curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash
# az login

## Step 6 — Load config + connect to your Foundry project (Python)
This cell loads `.env` and creates a Foundry project client.

In [4]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents import AgentsClient

load_dotenv()

project_endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"]
existing_agent_name = os.environ["PRODUCT_INVENTORY_MANAGER_AGENT_NAME"]
model_id = os.getenv("AZURE_FOUNDRY_PROJECT_MODEL_ID", "gpt-4o-mini")
# Get the MCP server configuration from environment variables
#mcp_server_url = os.environ.get("MCP_MSFT_DOCS_SERVER")
#mcp_server_label = os.environ.get("MCP_SERVER_LABEL", "msdocs")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
agent_client = AgentsClient( endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to project endpoint:", project_endpoint)
print("Reusing agent name:", existing_agent_name)
print("Default model id (for new agents):", model_id)


Connected to project endpoint: https://lapate-5938-resource.services.ai.azure.com/api/projects/lapate-5938
Reusing agent name: ProductInventoryManager
Default model id (for new agents): gpt-4.1


## Step 7 — Reuse the ProductInventoryManager agent by Name
This cell retrieves the persistent agent created in Lab 1.

In [6]:
# The projects client exposes agent operations via .agents
# Retrieve existing agent
product_inventory_agent = project_client.agents.get(agent_name=existing_agent_name)

print("Agent name:", getattr(product_inventory_agent, 'name', None))


Agent name: ProductInventoryManager


## Step 8 — Ask the existing agent a question
This uses the OpenAI-compatible Responses API through the Foundry project.

> Note: Depending on SDK version, you may use `project_client.inference.get_openai_client()` or a similar helper. If your installed SDK exposes a different helper, use the code sample generated by Foundry/VS Code for your agent.

In [7]:
# --- Minimal pattern using OpenAI-compatible client ---
# Handle SDK differences for obtaining the OpenAI-compatible client.
if hasattr(project_client, "inference") and hasattr(project_client.inference, "get_openai_client"):
    openai_client = project_client.inference.get_openai_client()
elif hasattr(project_client, "get_openai_client"):
    openai_client = project_client.get_openai_client()
else:
    candidates = [n for n in dir(project_client) if "openai" in n.lower() or "inference" in n.lower()]
    raise AttributeError(
        "Could not find an OpenAI client helper on project_client. "
        f"Found related attributes: {candidates}"
    )

question = "Describe our top-selling product and give a short product description."
product_inventory_agent = getattr(product_inventory_agent, "name", existing_agent_name)

response = openai_client.responses.create(
   input=question,
   extra_body={"agent": {"type": "agent_reference", "name": product_inventory_agent}},
)

print(response.output_text)

Based on your sales data, the top-selling product by revenue is the "Field Premium Extreme Temp Jacket" (SKU: ZCF-PR-ETJ), with total revenue of $488,459 and 1,903 units sold.

**Product Description:**
The Field Premium Extreme Temp Jacket is designed for high-performance outdoor activities in extreme weather conditions. This jacket combines advanced thermal insulation, windproof fabric, and durable construction to offer maximum comfort and protection for field athletes and outdoor enthusiasts. With a sleek design and premium materials, it is tailored for both professional use and casual wear, ensuring optimal warmth and mobility.

If you want descriptions for other top sellers, such as performance leggings or premium tops, let me know!


## Step 9 — Create a second agent: SalesInsightsAgent - We are using Agent Framework SDK to create all new agents in below steps
This cell creates a new persistent agent in your Foundry project.

## (Optional) — This is an optional step to delete any unwanted vector store files

In [8]:
# 1. Delete not needed a vector store for file search

vector_store = openai_client.vector_stores.list()
print(vector_store.data)
for id in vector_store.data:
    #print(f"Vector store ID: {id.id}, Name: {id.name}")
    if id.name.startswith('my_vector'):
        print(f"Found vector store with name 'my_vector_store_sales' and ID: {id.id}")
        openai_client.vector_stores.delete(vector_store_id=id.id)



[VectorStore(id='vs_GvKak1oiFsRac01d16TYAF3C', created_at=1772317265, file_counts=FileCounts(cancelled=0, completed=3, failed=0, in_progress=0, total=3), last_active_at=1772317265, metadata={}, name='my_vector_store_for_sales', object='vector_store', status='completed', usage_bytes=2236690, expires_after=None, expires_at=None), VectorStore(id='vs_7SmreVRQWQz7ek1sZgE1WFob', created_at=1772301922, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1772301922, metadata={}, name='index_serene_plow_q0175r397g', object='vector_store', status='completed', usage_bytes=4138, expires_after=None, expires_at=None)]
Found vector store with name 'my_vector_store_sales' and ID: vs_GvKak1oiFsRac01d16TYAF3C


## Step 9 — Create a vector store to serve the vector index for the filesearch tool (knowledge base for the agent)

In [9]:
# Create vector store for file search
vector_store = openai_client.vector_stores.create(name="my_vector_store_for_sales")
print(f"Vector store created (id: {vector_store.id})")

# Load the file to be indexed for search
asset_file_path = "./../../data/Zava Orders.txt"

# Upload file to vector store
file = openai_client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id, file=open(asset_file_path, "rb")
)
print(f"File uploaded to vector store (id: {file.id})")

# Create a tool. Use plain dict payloads for create_version (JSON-serializable)
tools = [{"type": "file_search", "vector_store_ids": [vector_store.id]}]


Vector store created (id: vs_YPgebwhICSUcvKs0RagVdUlW)
File uploaded to vector store (id: assistant-9QxnrYLyRqVxEXTaBKqsYR)


## Step 10 — Create a second agent: SalesInsightsAgent - We are using Agent Framework SDK to create all new agents in below steps
This cell creates a new persistent agent in your Foundry project.

In [10]:
from json import tool

from azure.ai.agents.models import FilePurpose, FileSearchTool
from azure.ai.agents.models import ToolDefinition, ToolResources
from azure.ai.projects.models import PromptAgentDefinition


# 1. Provide instructions for the agent that reference the tool's capabilities
sales_instructions = (
    "You are SalesInsightAgent. Translate inventory + customer insights into revenue risk/opportunity "
    "(upsell/cross-sell, replenishment urgency) and output a ranked list of top opportunities in JSON."
)


# 2. Create a new agent version with File Search tool enabled
sales_agent = project_client.agents.create_version(
    agent_name="SalesInsightsAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=sales_instructions,
        tools=tools,
    ),
    description="Agent that provides sales insights based on inventory and sales data.",
)

print("Created SalesInsightsAgent name:", getattr(sales_agent, "name", None))

Created SalesInsightsAgent name: SalesInsightsAgent


## Step 11 — Test SalesInsightsAgent
Ask a sales-focused question.

In [11]:

sales_agent_name = getattr(sales_agent, "name", "SalesInsightsAgent")
question = "Which products appear most popular and what are 3 talking points for leadership?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "name": sales_agent_name}},
)

print(response.output_text)

The most popular products, based on frequency across B2B/B2C orders in the data, are:

1. **Field Premium Performance Leggings**
2. **Field Elite Performance Leggings**
3. **Field Premium Long Sleeve Top**
4. **Field Elite Short Sleeve Top**
5. **Smart Cleats Pro/Standard**
6. **Field Standard Extreme Temp Jacket**

Three talking points for leadership:

### 1. Consistency across segments
- The Field Premium and Elite Performance leggings, as well as Smart Cleats (Pro and Standard), dominate both individual (B2C) and wholesale (B2B) channels. This consistency demonstrates broad market appeal and reliability across demographics.

### 2. Product line performance and upsell potential
- Top products lie in both "Field" apparel and "Smart" accessories, showing cross-category opportunity. There's a clear trend for premium versions (Premium/Elite) outperforming standard, supporting a higher price point and upsell strategy.

### 3. Inventory risk/opportunity for replenishment and cross-sell
- H

## Step 12 — Create customer insight agent

In [12]:
from azure.ai.projects.models import PromptAgentDefinition

# Load the file to be indexed for search
asset_file_path = "./../../data/Zava Customers.txt"

# Upload file to vector store
file = openai_client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id, file=open(asset_file_path, "rb")
)
print(f"File uploaded to vector store (id: {file.id})")

# Create a tool. Use plain dict payloads for create_version (JSON-serializable)
tools = [{"type": "file_search", "vector_store_ids": [vector_store.id]}]



# 1. Provide instructions for the agent that reference the tool's capabilities
customer_insight_instructions = (
                'You are CustomerInsightAgent. Given sales + inventory findings, '
                'infer customer segment impacts (e.g., high-value customers affected by stockouts), '
                'and propose 2-3 customer-facing insights in JSON.'
)


# 2. Create a new agent version with File Search tool enabled
customer_insight_agent = project_client.agents.create_version(
    agent_name="CustomerInsightAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=customer_insight_instructions,
        tools=tools,
    ),
    description="Agent that provides customer insights based on customer data.",
)

print("Created CustomerInsightAgent name:", getattr(customer_insight_agent, "name", None))

File uploaded to vector store (id: assistant-8eG3zb3WRV71gyZQSyFU53)
Created CustomerInsightAgent name: CustomerInsightAgent


## Step 13 — Test CustomerInsightsAgent
Ask a customer detail question.

In [13]:
customer_insight_agent = getattr(customer_insight_agent, "name", "CustomerInsightAgent")
question = "Given sales and inventory data, what customer segments are most impacted and what are 2-3 insights we can share with customers?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "name": customer_insight_agent}},
)

print(response.output_text)

Based on the sales and inventory findings, impacts on customer segments are as follows:

- **B2B segments** (such as Fabrikam, Wide World Importers, Contoso Ltd., Northwind Traders, etc.) represent the highest value customers with large order quantities and significant revenue, so they are most impacted by product shortages, stockouts, or delays. Their business operations depend heavily on timely delivery of raw materials and team uniforms.
- **Direct eCommerce (B2C)** is highly transactional with frequent, smaller purchases, but collectively, this segment accounts for considerable revenue and margin. B2C customers are more sensitive to product availability and may switch brands if their desired items are out of stock.
- **Regional variations:** East and West regions have especially high order volumes among B2B clients. Central and International regions see significant B2C activity, with demand spikes for accessories and premium/performance apparel.

**Customer-Facing Insights (JSON):*

## Step 14 — Create marketing action agent

In [14]:
from azure.ai.projects.models import PromptAgentDefinition

# Load the file to be indexed for search
asset_file_path = "./../../data/Zava Orders.txt"

# Upload file to vector store
file = openai_client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id, file=open(asset_file_path, "rb")
)
print(f"File uploaded to vector store (id: {file.id})")

# Create a tool. Use plain dict payloads for create_version (JSON-serializable)
tools = [{"type": "file_search", "vector_store_ids": [vector_store.id]}]



# 1. Provide instructions for the agent that reference the tool's capabilities
marketing_action_instructions = (
                'You are MarketingActionAgent. Propose marketing actions aligned to '
                'inventory reality (avoid promoting out-of-stock, push excess stock), '
                'and produce an action plan with channels, timing, and guardrails in JSON.'
)


# 2. Create a new agent version with File Search tool enabled
marketing_action_agent = project_client.agents.create_version(
    agent_name="MarketingActionAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=marketing_action_instructions,
        tools=tools,
    ),
    description="Agent that provides marketing actions based on inventory data.",
)

print("Created MarketingActionAgent name:", getattr(marketing_action_agent, "name", None))

File uploaded to vector store (id: assistant-7P3JmQtLi56LRmdCXP3PPa)
Created MarketingActionAgent name: MarketingActionAgent


## Step 15 — Test MarketingActionAgent
Ask a marketing detail question.

In [15]:
marketing_action_agent = getattr(marketing_action_agent, "name", "marketing_action_agent")
question = "Based on inventory and sales insights, what marketing actions should we take, and what channels and timing would you recommend?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "name": marketing_action_agent}},
)

print(response.output_text)

Based on recurring sales patterns and implied inventory movement in your orders data, here is a recommended marketing action plan aligned to inventory reality:

### 1. Push Products with Excess Inventory or Slow Sales:
- **Field Elite Performance Leggings (Bottoms)** and **Smart Cleats Pro (Footwear)** frequently appear in large B2B orders and smaller B2C sales, indicating strong supply or excess inventory risks.
- **Field Premium Long Sleeve Top**, **Field Standard Extreme Temp Jacket**, and **Field Elite Short Sleeve Top** are often ordered in small B2C increments, suggesting slower sales velocity on some channels.

**Actions:**
- Run targeted email campaigns focused on these SKUs, offering bundle discounts or seasonal promos.
- Launch paid social ads and retargeting to consumers who previously abandoned carts with these items.
- Consider affiliate partnerships or influencer promotions to move excess stock.
- Time promotions to coincide with relevant seasons (e.g., jackets in fall/wi

## Step 16 — Create a co-ordinator agent

In [16]:
from azure.ai.projects.models import PromptAgentDefinition
# 1. Provide instructions for the agent that reference the tool's capabilities
co_agent_instructions = (
                'Convert the user request into a structured brief '
                'for downstream agents.  '
                'goal, constraints, data_notes, required_outputs.'
)


# 2. Create a new agent version with File Search tool enabled

coagent = project_client.agents.create_version(
    agent_name="coagent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=co_agent_instructions,
    ),
    description="Agent that coordinates between different agents and produces a structured brief for them to execute against.",
)


print("Created coordination_agent name:", getattr(coagent, "name", None))

Created coordination_agent name: coagent


## Step 17 — Test co-odrination agent

In [ ]:
#Here No test is needed for co-agent as it is only responsible for structuring the user request into a brief for downstream agents. The effectiveness of the co-agent will be reflected in the performance of the downstream agents (SalesInsightsAgent, CustomerInsightAgent, MarketingActionAgent) when they execute against the structured brief provided by the co-agent.

## Step 18 —  Build the sequential workflow with WorkflowBuilder

We create a directed pipeline:

`ZavaCoordinatorAgent` → `ProductInventoryManagerAgent` → `CustomerInsightAgent` → `SalesInsightAgent` → `MarketingActionAgent`

The docs show the Python pattern `WorkflowBuilder(start_executor=...)` + `add_edge(...)` + `build()`.

### Agent	                        Responsibility

#### Inventory Intelligence Agent ----  Detects low‑stock risk, excess inventory, and fast movers
#### Customer Insight Agent ----------  Analyzes purchase patterns and lifecycle signals
#### Sales Opportunity Agent ---------  Identifies likely upsell / cross‑sell moments
#### Marketing Action Agent	----------  Proposes targeted campaigns or offers
#### Coordinator Agent ---------------  Delegates tasks and consolidates recommendations



In [18]:
from typing import cast
from agent_framework import WorkflowBuilder, Agent, AgentResponse
from agent_framework.azure import AzureAIAgentClient


prompt = (
    'Zava: Using our sales + product + customer dataset, produce a weekly brief: '
    '1) inventory risks (stockout/excess), 2) customer impact insights, '
    '3) sales opportunities, 4) marketing actions. Return JSON.'
)

product_inventory_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="ProductInventoryManager")
sales_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="SalesInsightsAgent")
customer_insight_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="CustomerInsightAgent")
marketing_action_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="MarketingActionAgent")
coagent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="coagent")

coagent = coagent_client.as_agent(name="coagent")   
product_inventory_agent = product_inventory_agent_client.as_agent(name="ProductInventoryManager")
sales_agent =  sales_agent_client.as_agent(name="SalesInsightsAgent")
customer_insight_agent = customer_insight_agent_client.as_agent(name="CustomerInsightAgent")
marketing_action_agent = marketing_action_agent_client.as_agent(name="MarketingActionAgent")

workflow = (
    WorkflowBuilder(start_executor=coagent)
    .add_edge(coagent, product_inventory_agent)
    .add_edge(product_inventory_agent, customer_insight_agent)
    .add_edge(customer_insight_agent, sales_agent)
    .add_edge(sales_agent, marketing_action_agent)     
    .build()
)
# Run the workflow with the user's initial message.
# For foundational clarity, use run (non streaming) and print the terminal event.
events = await workflow.run(prompt)

print('=== WORKFLOW OUTPUTS ===')
outputs = events.get_outputs()
# The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
# of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

# Summarize the final run state (e.g., COMPLETED)
print("Final state:", events.get_final_state())
print(events.get_outputs())




=== WORKFLOW OUTPUTS ===
coagent: Certainly! Here is a template JSON for your weekly brief. Please adapt the details based on your actual sales, product, and customer data.

```json
{
  "weekly_brief": {
    "date_range": "2024-06-03 to 2024-06-09",
    "inventory_risks": {
      "potential_stockouts": [
        {
          "product_id": "P2451",
          "product_name": "Vitamin D Tablets",
          "current_stock": 15,
          "average_weekly_sales": 22,
          "days_until_stockout": 4
        },
        {
          "product_id": "P1712",
          "product_name": "Pain Relief Gel",
          "current_stock": 30,
          "average_weekly_sales": 28,
          "days_until_stockout": 7
        }
      ],
      "potential_excess": [
        {
          "product_id": "P3120",
          "product_name": "First Aid Kit",
          "current_stock": 200,
          "average_weekly_sales": 7,
          "excess_stock_estimated_months": 7
        }
      ]
    },
    "customer_impact_insi

## Step 19 —  Run the workflow with your prompt - Simple Output

In [19]:
from typing import cast
from agent_framework import WorkflowBuilder, Agent, AgentResponse
from agent_framework.azure import AzureAIAgentClient
from agent_framework_orchestrations import SequentialBuilder


prompt = (
    'Zava: Using our sales + product + customer dataset, produce a weekly brief: '
    '1) inventory risks (stockout/excess), 2) customer impact insights, '
    '3) sales opportunities, 4) marketing actions. Return JSON.'
)

product_inventory_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="ProductInventoryManager")
sales_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="SalesInsightsAgent")
customer_insight_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="CustomerInsightAgent")
marketing_action_agent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="MarketingActionAgent")
coagent_client = AzureAIAgentClient(model_deployment_name=model_id, credential=credential, project_endpoint=project_endpoint, agent_name="coagent")

coagent = coagent_client.as_agent(name="coagent")   
product_inventory_agent = product_inventory_agent_client.as_agent(name="ProductInventoryManager")
sales_agent =  sales_agent_client.as_agent(name="SalesInsightsAgent")
customer_insight_agent = customer_insight_agent_client.as_agent(name="CustomerInsightAgent")
marketing_action_agent = marketing_action_agent_client.as_agent(name="MarketingActionAgent")


# Run the workflow with the user's initial message.
workflow = SequentialBuilder(participants=[coagent,product_inventory_agent, customer_insight_agent, sales_agent,marketing_action_agent]).build()

# 3) Treat the workflow itself as an agent for follow-up invocations
agent = workflow.as_agent(name="my_wk_agent")
agent_response = await agent.run(prompt)
print('=== WORKFLOW OUTPUTS ===')
if agent_response.messages:
    print("\n===== Conversation =====")
    for i, msg in enumerate(agent_response.messages, start=1):
        name = msg.author_name or msg.role
        print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")










Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x78739cad92d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x787392a644c0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7873927df880>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x787391b2c250>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x78739185ee30>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x78739307ab00>, 1814.59983092)])']
connector: <aiohttp.connector.TCPConnector object at 0x78739185edd0>


=== WORKFLOW OUTPUTS ===

===== Conversation =====
------------------------------------------------------------
01 [user]
Zava: Using our sales + product + customer dataset, produce a weekly brief: 1) inventory risks (stockout/excess), 2) customer impact insights, 3) sales opportunities, 4) marketing actions. Return JSON.
------------------------------------------------------------
02 [coagent]
Certainly! Here’s a template response for your weekly brief, using sales, product, and customer datasets. If you upload your data, I can tailor to your specifics—otherwise, here’s a realistic example format:

```json
{
  "week": "2024-06-07",
  "inventory_risks": {
    "potential_stockouts": [
      {
        "product_id": "1001",
        "product_name": "Zava Sleep Aid",
        "current_stock": 30,
        "forecasted_weekly_sales": 45,
        "risk_level": "High"
      },
      {
        "product_id": "2002",
        "product_name": "Zava Immunity Supplement",
        "current_stock": 80,
  

# Optional - Run the workflow with your prompt - This is an example with with "Handoff" scenario. THIS is a HOME exercise 
## Step 20 — Orchestrate both agents with Microsoft Agent Framework (Python)

In [ ]:
from agent_framework import WorkflowBuilder
from agent_framework.azure import AzureAIAgentClient

# Handoff Orchestration Pattern: Dynamic Routing Based on Agent Output
# Each agent can decide if the workflow should continue or skip to a specific agent

# Scenario 1: Emergency Stockout Handoff
# If ProductInventoryManager detects critical stockout, bypass customer insights 
# and go directly to marketing for immediate action

# Scenario 2: Conditional Customer Analysis
# If SalesInsightAgent identifies high-value opportunity, trigger deeper 
# customer segmentation before marketing

# Scenario 3: Marketing Action Approval Loop
# If MarketingActionAgent proposes high-budget campaign, handoff back to 
# coordinator for human approval before execution

# Example: Build workflow with conditional handoff logic
# You can implement handoff by:
# 1) Using WorkflowBuilder with conditional edges
# 2) Creating a custom executor that examines agent output and routes accordingly
# 3) Using the SequentialBuilder with decision points

print("""
Handoff Orchestration Use Cases for Your 5-Agent Workflow:

1. **Critical Inventory Alert Handoff**
    - ProductInventoryManager detects <10% stock on top SKU
    - Handoff: Skip CustomerInsight → Direct to MarketingAction
    - Action: Immediate promotion pause + supplier escalation

2. **High-Value Customer Opportunity Handoff**
    - CustomerInsightAgent identifies VIP segment affected by stockout
    - Handoff: Loop back to SalesInsightAgent for upsell alternatives
    - Action: Personalized offer generation before marketing

3. **Budget Approval Handoff**
    - MarketingActionAgent proposes campaign >$50k
    - Handoff: Return to Coordinator for human approval
    - Action: Pause workflow, await confirmation, then resume

4. **Data Quality Handoff**
    - Any agent encounters incomplete data
    - Handoff: Escalate to Coordinator with specific data request
    - Action: Human-in-the-loop for data validation

5. **Multi-Channel Conflict Handoff**
    - MarketingActionAgent detects conflicting campaigns
    - Handoff: Back to SalesInsightAgent for priority ranking
    - Action: Re-sequence marketing actions by revenue impact

To implement, use WorkflowBuilder.add_conditional_edge() or
create custom executor logic that examines agent_response.text
for keywords like 'CRITICAL', 'APPROVAL_REQUIRED', 'DATA_MISSING'
""")

# Conceptual code structure (requires custom executor implementation):
# workflow = (
#     WorkflowBuilder(start_executor=coagent)
#     .add_edge(coagent, product_inventory_agent)
#     .add_conditional_edge(
#         product_inventory_agent,
#         lambda response: marketing_action_agent if "CRITICAL_STOCKOUT" in response.text 
#         else customer_insight_agent
#     )
#     .add_edge(customer_insight_agent, sales_agent)
#     .add_edge(sales_agent, marketing_action_agent)
#     .add_edge(marketing_action_agent, coagent, condition="APPROVAL_REQUIRED")
#     .build()
# )

## Step 21 — Cleanup (optional but recommended)
If you created SalesInsightsAgent only for the lab, delete it to avoid resource sprawl/cost.

In [ ]:
# Delete the agent you created in this lab (optional)
# project_client.agents.delete(agent_id=sales_agent_id)
# print('Deleted SalesInsightsAgent:', sales_agent_id)

## Validation Checklist
- ✅ Connected to Foundry project from Python
- ✅ Reused ProductInventoryManager by ID
- ✅ Created and tested SalesInsightsAgent
- ✅ Ran a sequential workflow orchestrating both agents